# 00 - Esegui B1, B2 e A1 in fila

Kaggle dà a ogni notebook la sua cartella `/kaggle/working`: se B1, B2 e A1 girano come notebook
separati, A1 non vede gli output degli altri due. Questo notebook li esegue uno dopo l'altro nella
stessa sessione, così ognuno legge gli output del precedente da `dfl_outputs`.

| Passo | Modulo | Produce | Usato da |
|---|---|---|---|
| 1. B1 palla morta | `pipeline/deadball.py` | `<match>_deadball.npz` (`p_dead`) + modello in `models/` | B2, A1 |
| 2. B2 tocco + controllo | `pipeline/touch_control.py --dead auto` | `<match>_rule.npz` (`labels`, `p_home`) | A1 |
| 3. A1 decoder | `pipeline/decoder.py` | `<match>_decoded.npz` + tabella finale | — |

Tempo indicativo con le griglie complete: 30-45 minuti, senza GPU. Con `QUICK = True` molto meno.
Tutti i dataset di input devono essere **privati**. Solo partite DFL: nessun dato di BAR-ATH.

## 1. Setup
Stessi input dei singoli notebook: codice della repo, partite DFL convertite (`DFL-MAT-*.npz`) e, consigliato, `ball_runs.npz`. Vedi `notebooks/README.md`.

In [ ]:
import os, sys, subprocess, time
REPO = os.environ.get("MTAH_REPO") or next((p for p in ["/kaggle/input/hackaton-barcellona-code", "/kaggle/working/Hackaton_Barcellona", os.path.abspath("..")] if os.path.isdir(os.path.join(p, "pipeline"))), None)
assert REPO, "Imposta MTAH_REPO alla cartella della repo"
sys.path[:0] = [REPO, os.path.join(REPO, "pipeline"), os.path.join(REPO, "starter"), os.path.join(REPO, "evaluation")]

QUICK = False   # True = griglie ridotte, per una prova veloce

import dfl_paths
print("partite DFL:", len(dfl_paths.match_paths()), "in", dfl_paths.converted_dir())
print("output in:", dfl_paths.out_dir())
# i sottoprocessi devono scrivere e leggere nella stessa cartella
os.environ["DFL_OUT"] = dfl_paths.out_dir()
os.environ.setdefault("DEADBALL_MODELS", "/kaggle/working/models" if dfl_paths.on_kaggle() else os.path.join(REPO, "models"))

In [ ]:
def run(script, *args):
    cmd = [sys.executable, os.path.join(REPO, "pipeline", script), *args]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.run(cmd, cwd=REPO, env=os.environ.copy(), capture_output=True, text=True)
    print(p.stdout[-20000:])
    if p.returncode != 0:
        print(p.stderr[-5000:])
        raise RuntimeError("{} fallito (codice {})".format(script, p.returncode))
    print("--- {} finito in {:.1f} min
".format(script, (time.time() - t) / 60))

## 2. B1: rilevatore di palla morta
Addestra LightGBM lasciando fuori una partita alla volta e scrive `p_dead` per ogni partita. Da guardare: spellF1 a valle con la `D` del rilevatore contro la `D` = palla mancante.

In [ ]:
run("deadball.py", *(["--quick"] if QUICK else []))

## 3. B2: tocco + conferma del controllo
`--dead auto` usa la palla morta di B1 appena scritta. Da guardare: spellF1 di tocco + controllo contro il metodo 4 e il baseline, e i possessi predetti contro veri.

In [ ]:
run("touch_control.py", "--dead", "auto", *(["--quick"] if QUICK else []))

## 4. A1: decoder a segmenti
Legge da solo `p_dead` (B1) e `p_home` (B2). Nella riga `sources` deve comparire `p_dead: B1 file` e `p_home B2: yes`: se no, gli output dei passi precedenti non sono stati trovati.

In [ ]:
import importlib, decoder
importlib.reload(decoder)
res = decoder.evaluate_dfl()
res["summary"]

## 5. Cosa riportare al team
Solo numeri aggregati su DFL:

- tabella finale di A1 (spellF1 medio, min, max per metodo);
- riga `sources` (quali output sono stati usati, `ball_runs` misurato o placeholder);
- dai passi 2 e 3: spellF1 della palla morta di B1 e di tocco + controllo.

Differenze sotto circa 0,04 di spellF1 sono rumore.

In [ ]:
print("sources:", res["sources"])
print("parametri scelti nei fold:", sorted(set(res["params"].values())))
print("file prodotti:", sorted(os.listdir(dfl_paths.out_dir())))